# Ingesta: fuentes → esquema `raw` (DuckDB)

Carga los 5 archivos fuente en `warehouse.duckdb`, esquema `raw`.

- **Estrategia:** full refresh (`CREATE OR REPLACE`). La fuente es un snapshot completo, sin `updated_at` ni CDC.
- **CSV:** todo como `VARCHAR`. El casteo y la validación se hacen en dbt (staging), donde quedan visibles y testeados.
- **products.json:** un registro por producto, tal cual viene.
- **fx_rates.json:** una fila por respuesta de la API; `rates` se guarda como JSON para no depender de qué monedas trae cada respuesta.
- **Metadatos de carga:** `_loaded_at` y `_source_file` en todas las tablas.

## 1. Configuración

In [1]:
from pathlib import Path
import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "analytics_engineer_assets").exists())
DATA = ROOT / "analytics_engineer_assets"
DB = ROOT / "warehouse.duckdb"

print("ROOT:", ROOT)
print("DB:  ", DB)
print("Archivos:", sorted(f.name for f in DATA.iterdir()))

ROOT: c:\Users\Emiliano\Documents\GitHub\Senior Analytics Engineer challenge - ER
DB:   c:\Users\Emiliano\Documents\GitHub\Senior Analytics Engineer challenge - ER\warehouse.duckdb
Archivos: ['customers.csv', 'fx_rates.json', 'order_items.csv', 'orders.csv', 'products.json']


In [2]:
con = duckdb.connect(str(DB))
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

## 2. CSV del Sales DB (customers, orders, order_items)

In [3]:
CSV_SOURCES = {
    "customers": "customers.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
}

for table, fname in CSV_SOURCES.items():
    path = (DATA / fname).as_posix()
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.{table} AS
        SELECT *,
               current_timestamp AS _loaded_at,
               '{fname}'         AS _source_file
        FROM read_csv('{path}', header = true, all_varchar = true)
    """)
    print(f"raw.{table} cargada")

raw.customers cargada
raw.orders cargada
raw.order_items cargada


## 3. Catálogo de productos (products.json)

In [4]:
path = (DATA / "products.json").as_posix()
con.execute(f"""
    CREATE OR REPLACE TABLE raw.products AS
    SELECT *,
           current_timestamp AS _loaded_at,
           'products.json'   AS _source_file
    FROM read_json_auto('{path}')
""")
con.sql("SELECT * FROM raw.products LIMIT 5").show()

┌───────┬───────────────────────────────┬─────────────┬────────────────────────────────────────────────────────────────────────┬────────────┬──────────┬───────────────────────────────┬───────────────┐
│  id   │             name              │  category   │                              description                               │ base_price │ currency │          _loaded_at           │ _source_file  │
│ int64 │            varchar            │   varchar   │                                varchar                                 │   double   │ varchar  │   timestamp with time zone    │    varchar    │
├───────┼───────────────────────────────┼─────────────┼────────────────────────────────────────────────────────────────────────┼────────────┼──────────┼───────────────────────────────┼───────────────┤
│     1 │ Wireless Bluetooth Headphones │ Electronics │ Premium noise-cancelling wireless headphones with 30-hour battery life │      89.99 │ USD      │ 2026-09-18 22:06:03.138721+02 │ products.js

## 4. Tasas de cambio (fx_rates.json)

El archivo es un objeto con un array `responses`; cada elemento es una respuesta de la API (`base`, `date`, `rates`).

In [5]:
path = (DATA / "fx_rates.json").as_posix()
con.execute(f"""
    CREATE OR REPLACE TABLE raw.fx_rates AS
    WITH responses AS (
        SELECT unnest(from_json(json(content) -> 'responses', '["JSON"]')) AS resp
        FROM read_text('{path}')
    )
    SELECT resp ->> 'base'   AS base_currency,
           resp ->> 'date'   AS rate_date,
           resp -> 'rates'   AS rates,
           current_timestamp AS _loaded_at,
           'fx_rates.json'   AS _source_file
    FROM responses
""")
con.sql("SELECT * FROM raw.fx_rates").show()

┌───────────────┬────────────┬───────────────────────────────────────┬───────────────────────────────┬───────────────┐
│ base_currency │ rate_date  │                 rates                 │          _loaded_at           │ _source_file  │
│    varchar    │  varchar   │                 json                  │   timestamp with time zone    │    varchar    │
├───────────────┼────────────┼───────────────────────────────────────┼───────────────────────────────┼───────────────┤
│ USD           │ 2024-06-01 │ {"USD":1.0,"EUR":0.9285,"GBP":0.7911} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ EUR           │ 2024-06-01 │ {"USD":1.077,"EUR":1.0,"GBP":0.852}   │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ GBP           │ 2024-06-01 │ {"USD":1.2641,"EUR":1.1737,"GBP":1.0} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ USD           │ 2024-09-15 │ {"USD":1.0,"EUR":0.9042,"GBP":0.7628} │ 2026-09-18 22:06:03.402834+02 │ fx_rates.json │
│ EUR           │ 2024-09-15 │ {"USD":1.1059,"EU

## 5. Verificación

In [6]:
con.sql("""
    SELECT 'customers' AS tabla, count(*) AS filas FROM raw.customers
    UNION ALL SELECT 'orders',      count(*) FROM raw.orders
    UNION ALL SELECT 'order_items', count(*) FROM raw.order_items
    UNION ALL SELECT 'products',    count(*) FROM raw.products
    UNION ALL SELECT 'fx_rates',    count(*) FROM raw.fx_rates
""").show()

┌─────────────┬───────┐
│    tabla    │ filas │
│   varchar   │ int64 │
├─────────────┼───────┤
│ customers   │    50 │
│ orders      │   453 │
│ order_items │   363 │
│ products    │   100 │
│ fx_rates    │     5 │
└─────────────┴───────┘



## 6. Cerrar la conexión

**Importante:** DuckDB permite un solo proceso escribiendo a la vez. Si la conexión queda abierta, `dbt run` va a fallar con un error de *lock*.

In [7]:
con.close()
print("Conexión cerrada")

Conexión cerrada
